###Transform Drivers Data
1. Read bronze drivers table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake case
4. Concatenate name.givenName and name.familyName to create a new column called driver_name and transform the value to Title case
Rename columns to make more meanimgful
5. Remove duplicate records
6. Transform values of column nationality to title case
7. Write the transformed data to drivers table

In [0]:
%run ../00-Common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.drivers"
silver_table = f"{catalog_name}.{silver_schema}.drivers"

In [0]:
silver_table

In [0]:
from pyspark.sql import functions as f

In [0]:
drivers_df = spark.read.option("versionAsOf", 0).table(bronze_table)

In [0]:
drivers_dropped_df = drivers_df.drop("url")

In [0]:
drivers_renamed_df = drivers_dropped_df\
    .withColumnsRenamed({
        "driverId": "driver_id",
        "dateOfBirth": "date_of_birth"
    })

In [0]:
drivers_concatenated_df = (
    drivers_renamed_df\
        .withColumn(
            "driver_name",
            f.concat_ws(" ", f.initcap(f.concat_ws(" ", f.col("name.givenName"), f.col("name.familyName"))))
        )
        .drop("name")
        

)

In [0]:
display(drivers_concatenated_df)

In [0]:
drivers_distinct_df = drivers_concatenated_df.dropDuplicates(["driver_id"])

In [0]:
drivers_final_df = drivers_distinct_df\
    .withColumn("nationality", f.initcap(f.col("nationality")))

In [0]:
display(drivers_final_df)

In [0]:
(
    drivers_final_df
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))

In [0]:
%sql
select * from formula1.silver.drivers